In [ ]:
#INSTALL EVERYTHING
!pip install langchain langchain-community chromadb sentence-transformers fastapi uvicorn

In [ ]:
#CREATE CBT DATA
documents = [
    "If you feel anxious, try deep breathing exercises.",
    "Grounding technique: focus on 5 things you can see.",
    "If you feel overwhelmed, break tasks into small steps.",
    "You are not alone. Talking to someone can help.",
    "Practice mindfulness to reduce stress."
]

In [ ]:
#CREATE VECTOR DATABASE
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embedding = HuggingFaceEmbeddings()

db = Chroma.from_texts(documents, embedding)

In [ ]:
#TEST RAG
query = "I feel stressed"

results = db.similarity_search(query)

print("Response:", results[0].page_content)

In [ ]:
#SIMPLE API
from fastapi import FastAPI

app = FastAPI()

@app.get("/chat")
def chat(msg: str):
    result = db.similarity_search(msg)
    return {"response": result[0].page_content}

In [ ]:
print(db.similarity_search("I feel overwhelmed")[0].page_content)

In [ ]:
print(db.similarity_search("I feel stressed")[0].page_content)

In [ ]:
print(db.similarity_search("I feel anxious")[0].page_content)

**DAY - 3**

In [ ]:
#CRISIS DETECTION
def detect_crisis(text):
    text = text.lower()

    crisis_keywords = [
        "kill myself",
        "want to die",
        "suicide",
        "end my life",
        "no reason to live"
    ]

    for word in crisis_keywords:
        if word in text:
            return True

    return False

In [ ]:
print(detect_crisis("I want to die"))   # True
print(detect_crisis("hello"))          # False

In [ ]:
#SIMPLE SENTIMENT (USING TAGS FOR NOW)
def detect_sentiment(text):
    text = text.lower()

    if any(word in text for word in ["sad", "lonely", "depressed"]):
        return "sad"

    elif any(word in text for word in ["stress", "anxious", "worried"]):
        return "anxious"

    else:
        return "neutral"

In [ ]:
print(detect_sentiment("I feel sad"))
print(detect_sentiment("I am stressed"))
print(detect_sentiment("hello"))

In [ ]:
#TIER SYSTEM
def get_tier(text):
    if detect_crisis(text):
        return "Tier 3"

    sentiment = detect_sentiment(text)

    if sentiment in ["sad", "anxious"]:
        return "Tier 2"

    return "Tier 1"

In [ ]:
print(get_tier("I want to die"))     # Tier 3
print(get_tier("I feel sad"))        # Tier 2
print(get_tier("hello"))             # Tier 1

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import json

with open("intents.json") as f:
    data = json.load(f)

In [ ]:
print(data.keys())

In [ ]:
for intent in data['intents']:
    intent['responses'] = [
        response.replace("Pandora", "Naga AI")
        for response in intent['responses']
    ]

In [ ]:
#FINAL CHATBOT LOGIC
import random

def final_chatbot(text):
    tier = get_tier(text)

    #Tier 3 → emergency
    if tier == "Tier 3":
        return "Please contact a helpline immediately. You are not alone."

    #Tier 2 → use RAG (your db)
    elif tier == "Tier 2":
        result = db.similarity_search(text)
        return result[0].page_content

    #Tier 1 → normal chatbot (intents)
    else:
        for intent in data['intents']:
            for pattern in intent['patterns']:
                if pattern.lower() in text.lower():
                    return random.choice(intent['responses'])

        return "I'm here for you. Can you tell me more?"

In [ ]:
print(final_chatbot("hello"))
print(final_chatbot("I feel stressed"))
print(final_chatbot("I want to die"))

In [ ]:
#LIVE CHAT
while True:
    user = input("You: ")

    if user.lower() in ["exit", "quit", "bye"]:
        print("Naga AI: Goodbye!")
        break

    print("Naga AI:", final_chatbot(user))

In [ ]:
print("Test 1:", final_chatbot("hello"))
print("Test 2:", final_chatbot("I feel overwhelmed"))
print("Test 3:", final_chatbot("I want to die"))